# Limpieza de datos crudos IGDB (`juegos_raw`)

En este notebook se realiza la limpieza inicial del dataset de juegos
extraído desde la API de IGDB. Los objetivos principales son:

- Estandarizar tipos de datos.
- Convertir columnas que almacenan listas como texto a listas reales.
- Limpiar fechas de lanzamiento y restringir el rango de años.
- Filtrar tipos de juego relevantes (juegos principales y variantes completas).
- Eliminar registros duplicados y juegos claramente “ruido” (DLC, packs, etc.).
- Quitar columnas poco informativas o muy incompletas.


In [86]:
import pandas as pd
from pathlib import Path
from ast import literal_eval

In [ ]:
# Rutas a los datos crudos y a la carpeta de salida
ruta_no_limpios = Path("..") / ".." / "data" / "raw"
ruta_limpios = Path("..") / ".." / "data" / "processed"
ruta_limpios.mkdir(parents=True, exist_ok=True)

# Cargar dataset crudo de juegos
df_juegos_raw = pd.read_csv(ruta_no_limpios / "juegos_raw.csv")
df_juegos_raw.shape

(343323, 20)

## 2. Inspección inicial de valores faltantes

Primero revisamos el porcentaje de valores faltantes por columna
para tener una idea de qué tan completo está el dataset crudo.


In [73]:
df_juegos_raw.isna().mean().sort_values(ascending=False)

aggregated_rating          0.952103
aggregated_rating_count    0.952103
hypes                      0.947729
status                     0.925939
rating                     0.903988
rating_count               0.903988
total_rating_count         0.887989
total_rating               0.887989
keywords                   0.625469
player_perspectives        0.607649
involved_companies         0.521955
themes                     0.435039
game_modes                 0.364939
first_release_date         0.268199
platforms                  0.198810
genres                     0.169939
slug                       0.000003
name                       0.000003
id                         0.000000
game_type                  0.000000
dtype: float64

## 3. Conversión de columnas que representan listas

Varias columnas (`game_modes`, `genres`, `platforms`, etc.) vienen como
texto con formato de lista de Python (por ejemplo `"[1, 2, 3]"`).
Las convertimos a listas reales utilizando `literal_eval`.


In [88]:
df_juegos = df_juegos_raw.copy()

columnas_lista = [
    "game_modes",
    "genres",
    "involved_companies",
    "platforms",
    "themes",
    "keywords",
    "player_perspectives",
]

def convertir_a_lista_seguro(valor):
    """Convierte strings tipo '[1, 2]' a lista [1, 2]."""
    if pd.isna(valor):
        return []
    if isinstance(valor, list):
        return valor
    s = str(valor).strip()
    if not s:
        return []
    try:
        return literal_eval(s)
    except (SyntaxError, ValueError):
        return []

for col in columnas_lista:
    df_juegos[col] = df_juegos[col].apply(convertir_a_lista_seguro)

df_juegos[columnas_lista].head()

,game_modes,genres,involved_companies,platforms,themes,keywords,player_perspectives
0,[1],"[10, 14, 32]",[176923],[163],[1],[],[]
1,[1],"[31, 32]",[],[6],"[1, 19]",[131],[1]
2,[1],"[13, 32]",[347297],[6],[1],[],[]
3,[1],"[13, 15, 32]",[],[6],[1],[77],[3]
4,"[1, 2]","[5, 32]",[154804],[6],[1],[974],[1]


## 4. Limpieza de fechas de lanzamiento y rango de años

Convertimos el timestamp `first_release_date` a una fecha legible y
obtenemos el año de lanzamiento. Luego restringimos el dataset a juegos
lanzados entre 2000 y 2025.


In [89]:
# Conversión de timestamp a fecha y año
df_juegos["fecha_lanzamiento"] = pd.to_datetime(
    df_juegos["first_release_date"],
    unit="s",
    errors="coerce",
)
df_juegos["año_lanzamiento"] = df_juegos["fecha_lanzamiento"].dt.year

df_juegos[["first_release_date", "fecha_lanzamiento", "año_lanzamiento"]].head()
df_juegos["año_lanzamiento"].describe()

# Rango razonable para el análisis
df_juegos["fecha_valida"] = df_juegos["año_lanzamiento"].between(2000, 2025)
df_juegos = df_juegos[df_juegos["fecha_valida"]].copy()
df_juegos = df_juegos.drop(columns="fecha_valida")

df_juegos["año_lanzamiento"].describe()


count    220119.000000
mean       2018.223347
std           6.324487
min        2000.000000
25%        2015.000000
50%        2020.000000
75%        2023.000000
max        2025.000000
Name: año_lanzamiento, dtype: float64

## 5. Filtro por tipo de juego y eliminación de ruido

Nos quedamos solo con tipos de juego que representan productos completos
(main game, remakes, remasters, ports, etc.), excluyendo DLC, bundles,
mods y contenido claramente adicional.


In [90]:
# Tipos de juego que no son expansiones o contenido adicional
tipos_principales = [0, 4, 8, 9, 10, 11]

df_juegos = df_juegos[df_juegos["game_type"].isin(tipos_principales)].copy()
df_juegos["game_type"].value_counts(dropna=False)


game_type
0     180431
11      4319
10      1461
9       1109
8        988
4        350
Name: count, dtype: int64

### 5.2 Eliminación de duplicados por nombre + fecha

Si existen varias filas con el mismo `name` y la misma `first_release_date`,
nos quedamos con la que tiene mayor `total_rating_count`, asumiendo que
es la versión más informativa.


In [91]:
# Ordenar para que arriba quede la fila con más reviews
df_juegos = df_juegos.sort_values(
    by=["name", "first_release_date", "total_rating_count"],
    ascending=[True, True, False]
)

df_juegos = df_juegos.drop_duplicates(
    subset=["name", "first_release_date"],
    keep="first"
)

df_juegos.shape


(187927, 22)

### 5.3 Eliminación de juegos “ruido” (DLC, packs, demos, etc.)

Finalmente, filtramos juegos cuyo nombre contiene palabras clave que
indican que se trata de DLC, packs, demos, betas, bandas sonoras, etc.

In [92]:
def es_ruido_por_nombre(nombre):
    n = str(nombre).lower()
    palabras_ruido = [
        "dlc",
        "pack",
        "skin pack",
        "texture pack",
        "season pass",
        "episode",
        "ep.",
        "demo",
        "beta",
        "alpha",
        "trial",
        "soundtrack",
    ]
    return any(p in n for p in palabras_ruido)

mascara_ruido = df_juegos["name"].apply(es_ruido_por_nombre)
df_juegos = df_juegos[~mascara_ruido].copy()
df_juegos.shape


(186076, 22)

## 6. Eliminación de columnas poco informativas

Quitamos columnas muy incompletas o redundantes para el análisis
(`aggregated_rating`, `aggregated_rating_count`, `hypes`, `status`,
`first_release_date`, etc.).


In [93]:
columnas_a_dropear = [
    "aggregated_rating",
    "aggregated_rating_count",
    "status",
    "hypes",
    "first_release_date",  # ya usamos fecha_lanzamiento
]

df_juegos = df_juegos.drop(columns=columnas_a_dropear)
df_juegos.columns


Index(['id', 'game_modes', 'genres', 'involved_companies', 'name', 'platforms',
       'slug', 'themes', 'game_type', 'keywords', 'player_perspectives',
       'rating', 'rating_count', 'total_rating', 'total_rating_count',
       'fecha_lanzamiento', 'año_lanzamiento'],
      dtype='object')

## 7. Corrección de tipos numéricos y sanity check

Convertimos columnas de rating y conteos a tipos numéricos adecuados y
reemplazamos por `NaN` cualquier valor fuera de rango lógico
(ratings fuera de [0, 100], conteos negativos).


In [94]:
# Conversión de columnas numéricas

# ratings
for col in ["rating", "total_rating"]:
    if col in df_juegos.columns:
        df_juegos[col] = pd.to_numeric(df_juegos[col], errors="coerce")

# conteos de votos
for col in ["rating_count", "total_rating_count"]:
    if col in df_juegos.columns:
        df_juegos[col] = (
            pd.to_numeric(df_juegos[col], errors="coerce")
            .astype("Int64")
        )

# tipo de juego y año
df_juegos["game_type"] = df_juegos["game_type"].astype("Int64")
df_juegos["año_lanzamiento"] = df_juegos["año_lanzamiento"].astype("Int64")

# Limpieza de valores fuera de rango

# ratings fuera de 0–100 → NaN
for col in ["rating", "total_rating"]:
    if col in df_juegos.columns:
        df_juegos.loc[
            ~df_juegos[col].between(0, 100, inclusive="both"),
            col
        ] = pd.NA

# conteos negativos → NaN
for col in ["rating_count", "total_rating_count"]:
    if col in df_juegos.columns:
        df_juegos.loc[df_juegos[col] < 0, col] = pd.NA


## 8. Resumen final y guardado

Revisamos la estructura final del DataFrame limpio y lo guardamos en
`data/limpios/juegos_igdb_limpio.csv` para usarlo luego en el EDA y la
construcción del modelo de predicción.


In [95]:
df_juegos.info()
df_juegos.head()

# Guardar versión limpia
df_juegos.to_csv(ruta_limpios / "juegos_igdb_limpio.csv", index=False)


<class 'pandas.core.frame.DataFrame'>
Index: 186076 entries, 194678 to 113118
Data columns (total 17 columns):
 #   Column               Non-Null Count   Dtype         
---  ------               --------------   -----         
 0   id                   186076 non-null  int64         
 1   game_modes           186076 non-null  object        
 2   genres               186076 non-null  object        
 3   involved_companies   186076 non-null  object        
 4   name                 186075 non-null  object        
 5   platforms            186076 non-null  object        
 6   slug                 186075 non-null  object        
 7   themes               186076 non-null  object        
 8   game_type            186076 non-null  Int64         
 9   keywords             186076 non-null  object        
 10  player_perspectives  186076 non-null  object        
 11  rating               26228 non-null   float64       
 12  rating_count         26228 non-null   Int64         
 13  total_rating  